# Partial-front stages and one reflected-output union

**CPU only. Run all four code cells.** This notebook uses about 27.3 MB of
immutable archived predictions and labels. No photographs, model weights,
GPU, training or new neural inference are required. NumPy is the only
numerical dependency and is installed only if it is missing.

First, trace three restored partial-front misses through the saved stages
actually available for each checkpoint. Complete pre-confidence/pre-NMS
tensors exist only for the historical adapted original view. Newer caches
are already filtered; their limits remain explicit.

Then replay one fixed hypothesis: keep the completed original detections,
add all reflected outputs unused by the existing mutual associations,
and run the existing final suppression rule. The rule is fixed before
full scoring, uses no reference to choose additions, and is evaluated
across five methods and four references. The three focused restorations
cannot be recovered by this union; other missed fronts justify the probe.

This is development evidence. No thresholds, labels, scores, coordinates
or model weights are tuned, and no method is automatically promoted.


## 1 Load the immutable implementation

Source stays separate from cached inputs and generated outputs. A changed
source checkout is preserved and rejected; choose another
`BAOMAI_PARTIAL_DETECTION_BASE` to run beside an edited copy.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

SOURCE_COMMIT = "0e1f545a2cf33022b86bf316866b08cacfa55477"
SOURCE_HASHES = {'analyze_partial_detection_r68.py': '18a2f0f35d051d87fb158c3a0908e461cf85c420b8f02b59ebb42574984f2d5a', 'analyze_duplicate_audit_r66.py': '868ab2c5df33ba376da4212aca6281d99a9d5c3a9d6cd1dac7dfaf4e1cd7acfa', 'analyze_proposal_stages_r63.py': '0dd585cb7d957dcd599c0f40fbbf8fcc1ffc85c9bf98f98307eb0c799f29f15b', 'audit_additional_front_pilot_r45_results.py': '4f2d5ddf167ae480ede653fa821efd8b36c5679aed9d3e79e98b1d7f71f4de3b', 'audit_additional_front_pilot_r47_results.py': '6d7a2d3ece42b9eb26a473437a8f8377474dee25d4b83c14a80cb04ab03d64d9', 'audit_front_diagnosis_r49_results.py': '5bb583f1cd9adfbf1cbef5084df935a6c2b2587f5b6c730b49ea26029d1244e2', 'audit_reflection_return.py': '55f391535b040a27682e899aec74915b87864063023bc1ed818f8f82e67101de', 'diagnose_fronts_r50.py': 'cba4f1df6dcf431c933af1506a4de15718636d9477406eb34779ac4d8ab6ee3e', 'evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0', 'export_front_error_review_r48.py': '45f8e98b2ecd77fc5cdd2db8debb018dfa3904cd373b875049eb491eb62227b5', 'pose_checkpoint.py': 'c1311307fd49b37e60f2939c9b084bc36690c1d13b6b31b9e4c51ea9353562d4', 'pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018', 'predict_pose_folder.py': '2942864e86303a67911826fc0dbfe547b4b4fc2ed3860e74c9d0210c635520a6', 'replay_front_confidence_r51.py': '97828d8423f6b4f693a36c42bc23d1b868540bdfa6b75bfaf42373cffc5ba8c0', 'replay_partial_inclusion_r67.py': '5502022e8f5a7b510e875f2948d3dcbf4b1ab59addd94a9e5d3260ec3eda7f9a', 'run_corner_average_r58.py': '286232cf490f950b367068ea52b43078970ab27bb016169cb99159dab2fdbd88', 'run_flip_inference_r56.py': 'bf2ff9056f12976ca8c4aa971de2d2a7af1c679369c44dbf73de9f9d2542700d', 'run_inference_ablation_r53.py': '8c8a37c3dcaccde5c568e9df6f44651cdb05965ca5f5ed5b0f2978852387dcc1', 'run_original_comparison_r59.py': '279e2c0bd701299ea2a6f30403e97da68cc44a9c4ff874c5ee5a0d155a6e6ea3', 'run_reference_matrix_r60.py': '2939b2789abf2566d7680de60a0a87efae2a41e45c2dc01c9d15d1221cb8bbf1', 'run_training_label_comparison_r61.py': '0dc8d5dfdd344f590f88d17ea7ced76f3fc018026fae221a4df6e25d7c807a9c'}
BASE = Path(os.environ.get("BAOMAI_PARTIAL_DETECTION_BASE", "/content/baomai-partial-detection-r68")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
INPUT_DIR, OUTPUT = WORK / "inputs", WORK / "results"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0", "PYTHONDONTWRITEBYTECODE": "1"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Existing files preserved; choose a fresh BAOMAI_PARTIAL_DETECTION_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", "/scripts/", "/docs/thesis/evidence/annotation-reference/baomai-annotation-reference-r60-results.zip", "/docs/thesis/evidence/partial-inclusion-r67/baomai-partial-inclusion-r67-results.zip",
         "/data/reviews/fronts-2026-11-28-r67/", "/docs/thesis/evidence/partial-inclusion-r67/excluded-front-provenance.json", "/docs/thesis/evidence/partial-detection-r68/protocol.json"], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
def verify_source():
    if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
        raise RuntimeError("Source changed; choose a fresh BAOMAI_PARTIAL_DETECTION_BASE.")
    for name, expected in SOURCE_HASHES.items():
        if hashlib.sha256((SOURCE / "scripts" / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError("Implementation hash differs: " + name)
    if hashlib.sha256((SOURCE / "docs/thesis/evidence/partial-detection-r68/protocol.json").read_bytes()).hexdigest() != "14fe3559bb22a584c17e29364e2ce7a481a7b0136c884a16ec93385da2a1ae71":
        raise RuntimeError("Fixed protocol changed.")
verify_source()
WORK.mkdir(parents=True, exist_ok=True)
print("Immutable source:", SOURCE_COMMIT)


## 2 Verify the saved inputs and fixed protocol

Archive hashes, sizes, CRC, member inventories, reference lineage and
source hashes must agree. Cached files are reused. Historical labels
and training labels remain untouched. Existing compatible NumPy is used.


In [ ]:
import importlib.util
if importlib.util.find_spec("numpy") is None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "numpy==2.3.5"])
import numpy as np
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE / "scripts"))
import analyze_partial_detection_r68 as audit
audit.stage_inputs(SOURCE, INPUT_DIR)
protocol = json.loads((SOURCE / audit.PROTOCOL_PATH).read_bytes())
print("Verified immutable archives:", len(audit.INPUTS))
print("NumPy:", np.__version__)
print("Fixed comparison:", protocol["variant"], "across", len(protocol["methods"]), "methods and", len(protocol["references"]), "references")
print("Full raw pool: historical adapted original view only; current caches are post-native-NMS.")


## 3 Trace the focused misses and evaluate the fixed union

Reproduce archived baselines and the existing historical CPU/GPU identity
bridge. Report best-overlap box geometry, actual stage support and a
finite-pool confidence-only bound. Compare the focused sizes with the
unchanged training-label distribution, without equating size and visual
similarity.

The union adds only unused reflected outputs. An existing mutual box link
remains used when its corner pair is unavailable. Keep all original boxes,
scores and completed averaged corners, then suppress same-class overlap
strictly above 0.70 in score order. Fresh matching records additions,
removed originals, gains, losses, rematches and known-corner outcomes.


In [ ]:
summary, records = audit.analyze(SOURCE, INPUT_DIR)
audit.write_outputs(OUTPUT, summary, records, SOURCE)
from IPython.display import Markdown, display
display(Markdown((OUTPUT / "results.md").read_text()))
print("Fixed comparison rows:", len(summary["matrix"]))
print("Focused stage counts:", summary["focused_stage_counts"])
print("Evidence records:", summary["record_counts"])


## 4 Verify and package the evidence

Recheck the immutable source and every saved result member. The package
includes unchanged/union outputs, source-view identities, all four label
references, the complete comparison matrix, stage evidence and paired
change ledgers. This records one development experiment; it selects no
threshold or production method.


In [ ]:
verify_source()
RESULT = audit.package_results(OUTPUT, SOURCE_COMMIT)
manifest = {"source_commit": SOURCE_COMMIT, "notebook_code_cells": 4,
            "archived_baseline_replay": "exact", "fresh_inference": False,
            "training": False, "labels_changed": False, "threshold_search": False,
            "comparison_rows": len(summary["matrix"]), "automatic_promotion": False,
            "result": {"filename": RESULT.name, "bytes": RESULT.stat().st_size,
                       "sha256": hashlib.sha256(RESULT.read_bytes()).hexdigest()}}
(WORK / "package-manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))
from IPython.display import FileLink
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RESULT)))
else:
    files.download(str(RESULT))
